# 02 – Modeling & Evaluation (Phần 1: Baseline + k-NN + Decision Tree)
**Nhóm 8 – Đề tài 11: Dự báo nguy cơ bệnh tim mạch** | Nhánh: `feature/baseline-and-trees` | Phụ trách: Nguyễn Gia Long (022922)

Nội dung:
1. Đọc `data/cardio_cleaned.csv`, chia Train/Test 80/20 (`stratify=y`, `random_state=42`)
2. Mô hình sàn `DummyClassifier` + Baseline `LogisticRegression` (trong Pipeline) – **Checkpoint 1**
3. k-NN và Decision Tree (trong Pipeline), 5-Fold Stratified CV trên tập Train
4. Bảng so sánh, lưu `reports/results_part1.csv` để ghép vào bảng so sánh 5 mô hình ở Phần 2

> **Chống Data Leakage:** mọi phép scale/encode nằm trong `Pipeline` + `ColumnTransformer`, chỉ được `fit` trên fold train. Tập Test không dùng để chọn mô hình.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42          # GIỮ NGUYÊN ở mọi notebook của nhóm
TEST_SIZE = 0.20
N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)   # dùng chung cho mọi lần CV

## 1. Đọc dữ liệu & chia Train/Test

In [ ]:
# Tìm file dữ liệu (chạy từ thư mục notebooks/ hoặc gốc dự án)
CANDIDATES = [Path("../data/cardio_cleaned.csv"), Path("data/cardio_cleaned.csv")]
DATA_PATH = next(p for p in CANDIDATES if p.exists())

df = pd.read_csv(DATA_PATH)
print("Kích thước:", df.shape)
display(df.head())
df.info()

In [ ]:
# Biến mục tiêu: 'cardio' (1 = có bệnh tim mạch, 0 = không).
# Nếu file dữ liệu đặt tên cột mục tiêu khác, sửa TARGET ở đây.
TARGET = "cardio"
assert TARGET in df.columns, f"Không thấy cột '{TARGET}'. Cột hiện có: {list(df.columns)}"

# Bỏ cột không dùng làm đặc trưng:
#  - 'id': chỉ là mã định danh
#  - 'age': tuổi tính bằng NGÀY, trùng thông tin với 'age_years' (tránh đa cộng tuyến)
DROP_COLS = [c for c in ["id", "age"] if c in df.columns]
X = df.drop(columns=[TARGET] + DROP_COLS)
y = df[TARGET]

print("Tỷ lệ lớp:\n", y.value_counts(normalize=True).round(4))
print("Số giá trị thiếu:", int(X.isna().sum().sum()))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print("Tỷ lệ lớp Train/Test:", y_train.mean().round(4), "/", y_test.mean().round(4))

In [ ]:
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()   # cột chữ (bmi_category, bp_category, age_group)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cont_cols = [c for c in num_cols if X_train[c].nunique() > 10]
pass_cols = [c for c in num_cols if c not in cont_cols]
print("Liên tục (scale):", cont_cols)
print("Giữ nguyên:", pass_cols)
print("Categorical (one-hot):", cat_cols)

def make_preprocessor(scale=True):
    # scale=False cho Decision Tree (không cần chuẩn hoá); vẫn qua ColumnTransformer
    transformers = []
    if cont_cols:
        transformers.append(("num", StandardScaler() if scale else "passthrough", cont_cols))
    if pass_cols:
        transformers.append(("bin", "passthrough", pass_cols))
    if cat_cols:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols))
    return ColumnTransformer(transformers)

def make_pipeline(model, scale=True):
    return Pipeline([("prep", make_preprocessor(scale)), ("model", model)])

### Giải thích cách chia `cont_cols` và `pass_cols`
Quy tắc: cột số có `nunique() > 10` → `cont_cols` (scale bằng `StandardScaler`); còn lại → `pass_cols` (giữ nguyên).

| Nhóm | Cột | Xử lý | Lý do |
|---|---|---|---|
| `cont_cols` | height, weight, ap_hi, ap_lo, age_years, bmi, pulse_pressure, map | `StandardScaler` | Đơn vị và khoảng giá trị rất khác nhau (cm, kg, mmHg, năm...); bắt buộc scale cho Logistic và k-NN |
| `pass_cols` (nhị phân) | gender, smoke, alco, active | Giữ nguyên | Chỉ nhận 0/1 (gender 1/2), phương sai nhỏ |
| `pass_cols` (thứ hạng) | cholesterol, gluc (1–3), risk_score (0–6) | Giữ nguyên | Biến thứ hạng, ít giá trị khác nhau nên bị `nunique() > 10` xếp vào nhóm giữ nguyên |
| `cat_cols` | bmi_category, bp_category, age_group | `OneHotEncoder` | Biến chữ |

**Lưu ý với k-NN và Logistic Regression:** cholesterol, gluc và risk_score có khoảng giá trị lớn hơn biến 0/1, nên có thể ảnh hưởng nhiều hơn tới khoảng cách (k-NN) và hệ số (Logistic). Ô kiểm tra bên dưới đo độ lệch chuẩn và thử scale thêm 3 cột này để xem kết quả có thay đổi không. **Decision Tree** không bị ảnh hưởng vì cây chỉ so sánh ngưỡng, không phụ thuộc thang đo.

In [ ]:
# (a) Độ lệch chuẩn: cột liên tục sau scale có std = 1, để so với các cột giữ nguyên
print(X_train[pass_cols].std().round(3).sort_values(ascending=False))

# (b) Thử nghiệm độ nhạy: scale thêm 3 cột thứ hạng, so sánh ROC-AUC CV (5-fold, trên Train)
ordinal_cols = ["cholesterol", "gluc", "risk_score"]
def make_pipeline_scaled_ordinal(model):
    prep = ColumnTransformer([
        ("num", StandardScaler(), cont_cols + ordinal_cols),
        ("bin", "passthrough", [c for c in pass_cols if c not in ordinal_cols]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    return Pipeline([("prep", prep), ("model", model)])

sens = {}
for name, model in {"Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                    "k-NN (k=5)": KNeighborsClassifier(n_neighbors=5)}.items():
    auc_keep = cross_validate(make_pipeline(model), X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)["test_score"].mean()
    auc_scale = cross_validate(make_pipeline_scaled_ordinal(model), X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)["test_score"].mean()
    sens[name] = {"AUC (giữ nguyên 3 cột)": auc_keep, "AUC (scale thêm 3 cột)": auc_scale}
display(pd.DataFrame(sens).T.round(4))

## 2. Phần 1A – Mô hình sàn & Baseline Logistic Regression (Checkpoint 1)

In [ ]:
SCORING = {"accuracy": "accuracy", "precision": "precision", "recall": "recall",
           "f1": "f1", "roc_auc": "roc_auc"}

models = {
    "Dummy (most_frequent)": make_pipeline(DummyClassifier(strategy="most_frequent"), scale=False),
    "Logistic Regression":   make_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
}

## 3. Phần 1B – k-NN và Decision Tree
- **k-NN** (`k=5` mặc định, nhạy với thang đo nên bắt buộc scale).
- **Decision Tree** (`max_depth=5` làm baseline có kiểm soát; cây không giới hạn độ sâu gần như chắc chắn overfit. Siêu tham số sẽ được tinh chỉnh bằng GridSearchCV ở Phần 3).

In [ ]:
models["k-NN (k=5)"] = make_pipeline(KNeighborsClassifier(n_neighbors=5))
models["Decision Tree (depth=5)"] = make_pipeline(
    DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE), scale=False)

### Minh họa lý do chọn `max_depth=5` cho Decision Tree
Chạy thêm cây mặc định (`max_depth=None`, cây phát triển đến khi nút thuần) để đối chiếu **Train ROC-AUC** với **CV ROC-AUC**. Cấu hình này chỉ dùng để minh họa overfit, **không đưa vào** bảng kết quả `results_part1.csv`.

In [ ]:
tree_full = make_pipeline(DecisionTreeClassifier(max_depth=None, random_state=RANDOM_STATE), scale=False)
res_full = cross_validate(tree_full, X_train, y_train, cv=cv, scoring="roc_auc",
                          return_train_score=True, n_jobs=-1)
depth_demo = pd.DataFrame({
    "Train roc_auc": [res_full["train_score"].mean()],
    "CV roc_auc": [res_full["test_score"].mean()],
}, index=["Decision Tree (max_depth=None)"])
depth_demo["Gap"] = depth_demo["Train roc_auc"] - depth_demo["CV roc_auc"]
display(depth_demo.round(4))
# So sánh: Decision Tree (depth=5) ở bảng trên có Train ≈ CV (gap rất nhỏ) -> không overfit

### 5-Fold Stratified CV trên tập Train (không dùng Test)

In [ ]:
rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=SCORING,
                         return_train_score=True, n_jobs=-1)
    rows.append({
        "Model": name,
        **{f"CV {m}": res[f"test_{m}"].mean() for m in SCORING},
        "CV roc_auc (std)": res["test_roc_auc"].std(),
        "Train roc_auc": res["train_roc_auc"].mean(),
    })
cv_table = pd.DataFrame(rows).set_index("Model")
cv_table["Gap (train - CV) AUC"] = cv_table["Train roc_auc"] - cv_table["CV roc_auc"]
display(cv_table.round(4))

### Đo trên Test Set – **chỉ để báo cáo Checkpoint 1**
Không dùng số liệu này để chọn mô hình (việc chọn mô hình dựa trên CV ở trên; đánh giá cuối cùng trên Test thực hiện ở Phần 4).

In [ ]:
test_rows, fitted = [], {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)                      # fit chỉ trên Train
    fitted[name] = pipe
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    })
test_table = pd.DataFrame(test_rows).set_index("Model")
display(test_table.round(4))

In [ ]:
fig, axes = plt.subplots(1, len(fitted), figsize=(4.2 * len(fitted), 3.8))
for ax, (name, pipe) in zip(axes, fitted.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pipe.predict(X_test), ax=ax,
                                            cmap="Blues", colorbar=False)
    ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

## 4. Lưu kết quả Phần 1 để ghép Bảng 5 mô hình
File `reports/results_part1.csv` là đầu vào cho bảng so sánh 5 mô hình ở Phần 2 (các mô hình RF/Boosting lưu file tương ứng với **cùng tên cột**).

> **Lưu ý:** notebook đã `Clear All Outputs` nên thư mục `reports/` không có sẵn trong repo. Cell bên dưới tự tạo thư mục và file khi chạy. Ai làm nhánh khác cần số liệu Phần 1 thì **chạy notebook này trước** (hoặc lấy file `reports/results_part1.csv` đã commit trong nhánh này).

In [ ]:
base = Path("..") if Path("../data").exists() else Path(".")   # gốc dự án
out = base / "reports"
out.mkdir(parents=True, exist_ok=True)                         # tự tạo thư mục reports/ nếu chưa có
cv_table.round(4).to_csv(out / "results_part1.csv", encoding="utf-8-sig")
print("Đã lưu:", out / "results_part1.csv")

## 5. Nhận xét kết quả (5-Fold Stratified CV trên Train, 54.879 mẫu)
> **Kiểm tra khi chạy lại:** các số trong bảng Markdown dưới đây là số liệu đã điền tay sau khi chạy notebook (vì output đã bị xóa trước khi commit). Hãy chạy `Restart & Run All` một lần ở máy để xác nhận code chạy thông suốt và ra đúng các con số này (`random_state=42` nên kết quả lặp lại được).
| Mô hình | CV Accuracy | CV Recall | CV F1 | CV ROC-AUC | Gap AUC (train − CV) |
|---|---|---|---|---|---|
| Dummy | 0,5053 | 0,0000 | 0,0000 | 0,5000 | 0,0000 |
| Logistic Regression | 0,7297 | 0,6578 | 0,7065 | 0,7944 | 0,0006 |
| k-NN (k=5) | 0,6964 | 0,6846 | 0,6905 | 0,7442 | 0,1259 |
| Decision Tree (depth=5) | 0,7305 | 0,6709 | 0,7111 | 0,7905 | 0,0034 |

- **Dữ liệu cân bằng** (49,5% có bệnh) nên Dummy đạt Accuracy ≈ 0,505, ROC-AUC = 0,5 và Recall = 0: đây là mốc sàn để so sánh.
- **Logistic Regression** và **Decision Tree (depth=5)** gần như ngang nhau (AUC ≈ 0,79, Accuracy ≈ 0,73) và đều vượt xa baseline. Gap train–CV rất nhỏ → không overfit, độ lệch chuẩn giữa các fold nhỏ (≈ 0,004) → kết quả ổn định.
- **k-NN** kém hơn rõ về ROC-AUC (0,744) và có Gap = 0,126 → overfit (nhớ dữ liệu train). Recall của k-NN cao nhất (0,685) nhưng đổi lại Precision thấp hơn.
- Kết quả trên Test khớp CV (ví dụ Logistic: AUC 0,7950 vs CV 0,7944) → không có dấu hiệu rò rỉ dữ liệu.
- Recall mới ~0,66–0,68: còn bỏ sót khoảng 1/3 bệnh nhân. Hướng cải thiện: mô hình ensemble/boosting (Phần 2), tinh chỉnh siêu tham số (Phần 3) và hạ ngưỡng quyết định (Phần 4).
- **Vì sao chọn `max_depth=5`:** cây không giới hạn độ sâu (`max_depth=None`) có Train ROC-AUC = 1,000 nhưng CV ROC-AUC chỉ 0,634 (gap 0,366) → overfit nặng. Giới hạn depth=5 cho CV AUC 0,791 với gap 0,003.
- **Về 3 cột thứ hạng giữ nguyên (cholesterol, gluc, risk_score):** độ lệch chuẩn của chúng (0,57–1,03) cùng bậc với cột liên tục sau scale (=1). Scale thêm 3 cột này gần như không đổi kết quả (Logistic: 0,7944 → 0,7944; k-NN: 0,7442 → 0,7445), nên giữ nguyên là hợp lý.
- Kết luận Phần 1: đưa **Logistic Regression** và **Decision Tree** vào so sánh 5 mô hình; việc chọn Top 2 cuối cùng thực hiện ở Phần 2 khi có đủ 5 mô hình.